In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from itertools import islice
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from scipy.sparse import save_npz

/home/hussain/projects/spam-email-classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Preprocessing

In [2]:
# Set the path to the file you'd like to load
file_path = "spam_ham_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "venky73/spam-mails-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

/tmp/ipykernel_3768/1896723840.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [3]:
df.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [4]:
df.shape

(5171, 4)

In [5]:
df["text"][3]

'Subject: photoshop , windows , office . cheap . main trending\r\nabasements darer prudently fortuitous undergone\r\nlighthearted charm orinoco taster\r\nrailroad affluent pornographic cuvier\r\nirvin parkhouse blameworthy chlorophyll\r\nrobed diagrammatic fogarty clears bayda\r\ninconveniencing managing represented smartness hashish\r\nacademies shareholders unload badness\r\ndanielson pure caffein\r\nspaniard chargeable levin\r\n'

In [6]:
df.isna().sum()

Unnamed: 0    0
label         0
text          0
label_num     0
dtype: int64

In [7]:
df.columns

Index(['Unnamed: 0', 'label', 'text', 'label_num'], dtype='str')

In [8]:
df = df.drop(columns=["Unnamed: 0", "label"])

In [9]:
df.head()

,text,label_num
0,Subject: enron methanol ; meter # : 988291\r\n...,0
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\r\nthis deal is t...,0


In [10]:
df.duplicated().sum()

np.int64(178)

In [11]:
df.shape

(5171, 2)

In [12]:
df = df.dropna().drop_duplicates(subset="text").reset_index(drop=True)

In [13]:
df.shape

(4993, 2)

In [14]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/hussain/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to /home/hussain/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [16]:
STOP_WORDS = set(stopwords.words("english"))
list(islice(STOP_WORDS, 4))

["i'd", 'of', "it'd", 'yourself']

In [17]:
lemmatizer = WordNetLemmatizer()

In [18]:
def clean_text(text):
    text = re.sub(r"\\[rnt]", " ", text)           # literal \r \n \t
    text = re.sub(r"<[^>]+>", " ", text)           # HTML tags (before punctuation)
    text = text.lower()
    text = re.sub(r"^\s*subject\s*:", " ", text)   # leading "Subject:"
    text = re.sub(r"[^a-z0-9\s]", " ", text)          # punctuation, digits, symbols
    text = re.sub(r"\s+", " ", text).strip()       # collapse whitespace
 
    tokens = [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in STOP_WORDS and (len(w) > 2 or w.isdigit())
    ]
    return " ".join(tokens)

In [19]:
df["clean_text"] = df["text"].apply(clean_text)

In [20]:
# Drop emails that became empty after cleaning
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

In [21]:
df.head()

,text,label_num,clean_text
0,Subject: enron methanol ; meter # : 988291\r\n...,0,enron methanol meter 988291 follow note gave m...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0,hpl nom january 9 2001 see attached file hplno...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0,neon retreat around wonderful time year neon l...
3,"Subject: photoshop , windows , office . cheap ...",1,photoshop window office cheap main trending ab...
4,Subject: re : indian springs\r\nthis deal is t...,0,indian spring deal book teco pvr revenue under...


In [22]:
for i in range(3):
    print(f"ORIGNAL: {df["text"][i][:200]}")
    print(f"CLEANED: {df["clean_text"][i][:200]}")

ORIGNAL: Subject: enron methanol ; meter # : 988291
this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary
flow data provided by daren } .
please override pop ' s daily volume { pre
CLEANED: enron methanol meter 988291 follow note gave monday 4 3 00 preliminary flow data provided daren please override pop daily volume presently zero reflect daily activity obtain gas control change needed 
ORIGNAL: Subject: hpl nom for january 9 , 2001
( see attached file : hplnol 09 . xls )
- hplnol 09 . xls
CLEANED: hpl nom january 9 2001 see attached file hplnol 09 xl hplnol 09 xl
ORIGNAL: Subject: neon retreat
ho ho ho , we ' re around to that most wonderful time of the year - - - neon leaders retreat time !
i know that this time of year is extremely hectic , and that it ' s tough to
CLEANED: neon retreat around wonderful time year neon leader retreat time know time year extremely hectic tough think anything past holiday life past week december 25 january 1 like think minute

### Feature Extraction via TF-IDF

**Splitting Data**

In [23]:
X = df["clean_text"]
y = df["label_num"]

In [24]:
# 60% train, 40% temporary holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

In [25]:
# Split the 40% holdout in half: 20% validation (CV), 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [26]:
print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))
print("Spam ratio -> train: %.3f, val: %.3f, test: %.3f"
      % (y_train.mean(), y_val.mean(), y_test.mean()))

Train: 2994 | Val: 998 | Test: 999
Spam ratio -> train: 0.293, val: 0.293, test: 0.292


In [30]:
# create the vectorizer and fit it to the training data
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    )

In [31]:
X_train = vectorizer.fit_transform(X_train)
X_val = vectorizer.transform(X_val) 
X_test = vectorizer.transform(X_test)

In [32]:
# inspect
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
features = vectorizer.get_feature_names_out()
print("Sample features:", features[:20])
print("'dollar' in vocab:", "dollar" in features)

Train: (2994, 5000) | Val: (998, 5000) | Test: (999, 5000)
Sample features: ['00' '00 00' '00 20' '00 25' '00 30' '00 31' '00 32' '00 463' '00 90'
 '00 daren' '00 deal' '00 delivery' '00 nomination' '00 please' '00 price'
 '00 production' '00 save' '00 thru' '000' '000 00']
'dollar' in vocab: True
